# Proteus — Phase 1 Cloud Bootstrap

Run this notebook on a cloud GPU runtime (Colab T4 16GB / Kaggle P100 16GB / RunPod A6000) to execute Phase 1 end-to-end:

1. Smoke test on `google/gemma-2-2b-it`
2. **Generate IT-model entity prompts** (upstream ships these only for base models — must be regenerated for the IT variant)
3. Cache entity-token activations (Wikidata sweep across player/movie/city/song)
4. Cache random-token activations (Pile baseline)
5. Compute SAE latent separation scores (Ferrando feature analysis)
6. Steering experiments (causal verification of top latents)

**Prerequisites**

- A Hugging Face account with the [Gemma 2 license accepted](https://huggingface.co/google/gemma-2-2b-it)
- An HF read-scope token. In Colab: add it to `Secrets` as `HF_TOKEN`. In Kaggle: add it via `Add-ons → Secrets` as `HF_TOKEN`. Locally: `export HF_TOKEN=...`
- A GPU runtime selected (Colab: `Runtime → Change runtime type → T4 GPU`)
- The Proteus repo accessible. Update `REPO_URL` below to your fork.

**Wall clock**: the prompt-generation pass and the entity activation cache are the two heavy steps. Budget 8–14 hours on a T4, 3–6 on an A6000.

## 0. Detect runtime and clone the repo

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# === Update this if you fork the repo ===
REPO_URL = "https://github.com/AntitheticalElysium/Proteus.git"
REPO_DIR = Path("/content/Proteus") if "google.colab" in sys.modules else Path.cwd() / "Proteus"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--recurse-submodules", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "submodule", "update", "--init", "--recursive"], check=True)

os.chdir(REPO_DIR)
print(f"CWD: {Path.cwd()}")
subprocess.run(["ls", "-la"], check=True)

## 1. Install `uv` and sync the environment

Resolves the 249-package dependency tree from `pyproject.toml` (locked in `uv.lock`) and installs into a project-local `.venv`. First-run download is ~6 GB of CUDA wheels and takes 5–10 minutes on Colab; subsequent runs reuse the cache.

In [ ]:
# Install uv (https://docs.astral.sh/uv/)
subprocess.run(
    "curl -LsSf https://astral.sh/uv/install.sh | sh",
    shell=True, check=True,
)
os.environ["PATH"] = f"{Path.home() / '.local/bin'}:{os.environ['PATH']}"
subprocess.run(["uv", "--version"], check=True)

In [ ]:
# Sync the locked environment. Honors uv.lock for reproducibility.
subprocess.run(["uv", "sync", "--frozen"], check=True)

## 2. Hugging Face authentication

Loads `HF_TOKEN` from the runtime's secret store and logs in non-interactively.

In [ ]:
hf_token = None

# Colab: read from userdata
try:
    from google.colab import userdata  # type: ignore
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    pass

# Kaggle: read from kaggle_secrets
if hf_token is None:
    try:
        from kaggle_secrets import UserSecretsClient  # type: ignore
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass

# Fallback to env var
if hf_token is None:
    hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    raise RuntimeError(
        "HF_TOKEN not found. Set it in Colab Secrets / Kaggle Secrets / shell env, then re-run this cell."
    )

os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
subprocess.run(
    ["uv", "run", "huggingface-cli", "login", "--token", hf_token, "--add-to-git-credential"],
    check=True,
)
print("HF login OK")

## 3. Smoke test

Loads `google/gemma-2-2b-it`, runs one forward pass with a residual hook on layer 9, and one short greedy generation. **STOP HERE if this fails** — debug model loading and HF auth before kicking off the long cache jobs.

In [ ]:
subprocess.run(["uv", "run", "python", "scripts/00_smoke_test.py"], check=True)

## 5. Cache entity-token activations (Wikidata sweep)

Iterates over `player`, `movie`, `city`, `song` entity types using the prompts directory generated in step 4. Saves memory-mapped float32 activations under `vendor/sae_entities/dataset/cached_activations/entity/gemma-2-2b-it_wikidata_<entity>/`.

**Expected duration on T4: ~2–4 hours.** Watch `nvidia-smi` during the first batch to confirm `--batch_size 16` fits comfortably; bump to 32 in `scripts/01_cache_entity_acts.sh` if peak is well under 12 GB.

In [ ]:
# Set MAX_QUERIES to a small number (e.g. 200) for a quick pipeline-plumbing check.
# Leave unset for the full Phase 1 run.
gen_env = os.environ.copy()
# gen_env["MAX_QUERIES"] = "200"
subprocess.run(["bash", "scripts/00b_generate_entity_prompts.sh"], check=True, env=gen_env)

## 6. Cache random-token activations (Pile baseline)

Used by feature analysis to filter latents that fire on generic-language tokens. **Expected duration: ~1–2 hours.**

In [ ]:
subprocess.run(["bash", "scripts/01_cache_entity_acts.sh"], check=True)

## 7. Feature analysis — separation scores per layer

Computes SAE latent separation scores between known and unknown entities, identifies general latents that fire across all four entity types, and produces the headline per-layer plots.

**Verification gate**: separation scores should peak in middle layers (~L8–L14).

In [ ]:
subprocess.run(["bash", "scripts/02_cache_pile_acts.sh"], check=True)

## 8. Steering experiments

Applies the top known/unknown latents from step 7 as additive steering vectors at coefficients 20–400. Compares refusal rates between original and steered generations.

**Verification gate**: at least one (latent, coefficient) combination must induce refusal on previously-answered known entities, AND at least one combination must suppress refusal on unknown entities.

In [ ]:
subprocess.run(["bash", "scripts/03_feature_analysis.sh"], check=True)

## 9. Persist outputs

On Colab, the runtime is ephemeral — copy results to Google Drive or download them locally. The cached activations are large (~10–30 GB total) and are usually discarded after analysis; the separation-score CSVs and plots are the artifacts worth preserving.

In [ ]:
subprocess.run(["bash", "scripts/04_steering.sh"], check=True)

## 8. Persist outputs

On Colab, the runtime is ephemeral — copy results to Google Drive or download them locally. The cached activations are large (~10–30 GB total) and are usually discarded after analysis; the separation-score CSVs and plots are the artifacts worth preserving.

In [ ]:
# Tar up the lightweight outputs (plots, scores, latent rankings) for download.
import shutil
out_archive = REPO_DIR / "phase1_results.tar.gz"
subprocess.run(
    [
        "tar", "czf", str(out_archive),
        "-C", str(REPO_DIR / "vendor" / "sae_entities"),
        "--exclude=cached_activations",
        "dataset",
        "mech_interp",
    ],
    check=False,  # don't crash if some paths are missing
)
print(f"Wrote: {out_archive} ({out_archive.stat().st_size / 1e6:.1f} MB)" if out_archive.exists() else "Archive not produced")